# 8.1 Training and deploying a Scikit-learn model with SageMaker SDK v3

This example submits a SageMaker training job, uploads training data to S3, and creates a hosted endpoint. It requires AWS credentials, an execution role ARN, and a configured AWS region. Training and endpoint hosting can incur AWS charges.

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.training.configs import Compute, SourceCode
from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.spec.inference_spec import InferenceSpec
from sagemaker.serve.utils.types import ModelServer
from sagemaker.train.configs import InputData
from sagemaker.train.model_trainer import ModelTrainer
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split


# Replace this placeholder with the SageMaker execution role ARN for your account.
role_arn = "arn:aws:iam::<account-id>:role/<SageMakerExecutionRole>"
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
if region is None:
    raise ValueError("Configure an AWS region before using SageMaker.")

# SageMaker SDK v3 supports the Scikit-learn 1.9-0 training and inference image.
sklearn_training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.9-0",
    py_version="py3",
    instance_type="ml.m5.large",
    image_scope="training",
)
sklearn_inference_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.9-0",
    py_version="py3",
    instance_type="ml.m5.large",
    image_scope="inference",
)

# Prepare the Iris dataset
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data,
    iris.target,
    test_size=0.2,
    random_state=42,
)

feature_names = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
]
train_data = pd.DataFrame(X_train, columns=feature_names)
train_data["target"] = y_train

# upload_data expects a local file path, not the CSV text returned by to_csv().
with tempfile.TemporaryDirectory() as temp_dir:
    training_data_path = Path(temp_dir) / "iris_train.csv"
    train_data.to_csv(training_data_path, index=False)
    train_data_s3 = sagemaker_session.upload_data(
        path=str(training_data_path),
        key_prefix="sagemaker/sklearn-iris",
    )

project_dir = Path.cwd()
if not (project_dir / "iris_train.py").is_file():
    project_dir = project_dir / "08_ml_in_cloud_and_edge"
if not (project_dir / "iris_train.py").is_file():
    raise FileNotFoundError("Could not find iris_train.py beside this notebook.")

# Configure and start the SageMaker training job.
sklearn_trainer = ModelTrainer(
    role=role_arn,
    sagemaker_session=sagemaker_session,
    training_image=sklearn_training_image,
    source_code=SourceCode(
        source_dir=str(project_dir),
        entry_script="iris_train.py",
        ignore_patterns=[
            ".env",
            ".git",
            "__pycache__",
            ".DS_Store",
            ".cache",
            ".ipynb_checkpoints",
            ".venv",
            "*.ipynb",
        ],
    ),
    compute=Compute(
        instance_count=1,
        instance_type="ml.m5.large",
    ),
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=train_data_s3,
        )
    ],
    hyperparameters={
        "max_depth": 5,
        "n_estimators": 100,
    },
)
sklearn_trainer.train()


class IrisInferenceSpec(InferenceSpec):
    def load(self, model_dir):
        import joblib

        return joblib.load(Path(model_dir) / "model.joblib")

    def invoke(self, input_object, model):
        features = np.asarray(input_object, dtype=float)
        return model.predict(features).tolist()


# Build and deploy the trained model using the v3 ModelBuilder API.
model_builder = ModelBuilder(
    model=sklearn_trainer,
    inference_spec=IrisInferenceSpec(),
    schema_builder=SchemaBuilder(
        sample_input=[[5.1, 3.5, 1.4, 0.2]],
        sample_output=[0],
    ),
    image_uri=sklearn_inference_image,
    model_server=ModelServer.MMS,
    role_arn=role_arn,
    sagemaker_session=sagemaker_session,
)
model_builder.build()
endpoint = model_builder.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
)

# Invoke the endpoint with a few test examples.
test_data = X_test[:5].tolist()
try:
    response = endpoint.invoke(
        body=json.dumps(test_data),
        content_type="application/json",
    )
    predictions = json.loads(response.body.read().decode("utf-8"))
    print(f"Predictions: {predictions}")
finally:
    # Delete the endpoint even if inference fails to stop endpoint hosting charges.
    endpoint.delete()

# 8.2 Training a TensorFlow model with Vertex AI

This is an SDK example, not a local training example. Replace the project, bucket, package, and container image placeholders with real resources before running it. The training package must read the managed dataset, apply the supplied split arguments, and save its model under `AIP_MODEL_DIR`. Use currently supported Vertex AI TensorFlow training and serving images with compatible TensorFlow versions.

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd
from google.cloud import aiplatform, storage
from sklearn.datasets import load_iris


# Replace these example names with a real project and globally unique bucket.
project_id = "your-google-cloud-project"
bucket_name = "your-globally-unique-ml-bucket"
region = "us-central1"
staging_bucket_uri = f"gs://{bucket_name}"

storage_client = storage.Client(project=project_id)
aiplatform.init(
    project=project_id,
    location=region,
    staging_bucket=staging_bucket_uri,
)

bucket = storage_client.bucket(bucket_name)
if not bucket.exists():
    bucket = storage_client.create_bucket(bucket_name, location=region)

# Create a local example CSV and upload it to Cloud Storage.
iris = load_iris(as_frame=True)
training_data = iris.frame
with tempfile.TemporaryDirectory() as temp_dir:
    training_data_path = Path(temp_dir) / "iris.csv"
    training_data.to_csv(training_data_path, index=False)
    training_data_blob = bucket.blob("training-data/iris.csv")
    training_data_blob.upload_from_filename(str(training_data_path))

training_data_uri = f"gs://{bucket_name}/training-data/iris.csv"
dataset = aiplatform.TabularDataset.create(
    display_name="iris-dataset",
    gcs_source=[training_data_uri],
)

# These must point to an uploaded Python package and compatible supported images.
python_package_gcs_uri = f"gs://{bucket_name}/packages/tensorflow_trainer.tar.gz"
training_container_uri = "<supported-vertex-ai-tensorflow-training-image>"
serving_container_uri = "<compatible-vertex-ai-tensorflow-serving-image>"
job = aiplatform.CustomPythonPackageTrainingJob(
    display_name="iris-tensorflow-training",
    python_package_gcs_uri=python_package_gcs_uri,
    python_module_name="trainer.task",
    container_uri=training_container_uri,
    model_serving_container_image_uri=serving_container_uri,
)

# The custom training module must parse these arguments and split the data itself.
training_args = [
    "--training-fraction-split",
    "0.8",
    "--validation-fraction-split",
    "0.1",
    "--test-fraction-split",
    "0.1",
]
model = job.run(
    dataset=dataset,
    args=training_args,
    replica_count=1,
    machine_type="n1-standard-4",
    model_display_name="iris-tensorflow-model",
    sync=True,
)
print(f"Model training completed: {model.resource_name}")

endpoint = model.deploy(
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=2,
    sync=True,
)
print(f"Model deployed to endpoint: {endpoint.resource_name}")

# Undeploy the model and delete the endpoint after testing.
endpoint.undeploy_all()
endpoint.delete()

# 8.3 Training and deploying a Scikit-learn model with Azure Machine Learning SDK v2

This example uses the current Azure ML SDK v2. Replace the workspace and compute placeholders with your own resources before running it. Submitting the job and creating the managed online endpoint use Azure resources and can incur charges. The endpoint is deleted at the end of the example.

In [ ]:
import json
import tempfile
import uuid
from pathlib import Path

import pandas as pd
from azure.ai.ml import Input, MLClient, Output, command
from azure.ai.ml.entities import (
    CodeConfiguration,
    Environment,
    ManagedOnlineDeployment,
    ManagedOnlineEndpoint,
    Model,
)
from azure.identity import DefaultAzureCredential
from sklearn.datasets import load_iris


# Reads subscription, resource group, and workspace from config.json.
# Azure authentication is required to submit the job and create the endpoint.
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

project_dir = Path.cwd()
if not (project_dir / "azure_train.py").is_file():
    project_dir = project_dir / "08_ml_in_cloud_and_edge"
if not (project_dir / "azure_train.py").is_file():
    raise FileNotFoundError("Could not find azure_train.py beside this notebook.")

# Create the Iris CSV used as the job's input data.
iris = load_iris(as_frame=True)
training_data = iris.frame

# Define a reusable environment for training and serving.
sklearn_environment = Environment(
    name="iris-sklearn-environment",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file={
        "name": "iris-sklearn-env",
        "channels": ["conda-forge"],
        "dependencies": [
            "python=3.10",
            "pip",
            {
                "pip": [
                    "scikit-learn==1.7.2",
                    "pandas",
                    "joblib",
                    "azureml-inference-server-http",
                ]
            },
        ],
    },
)

# Create and submit a command job. The compute target must already exist.
compute_name = "cpu-cluster"  # Replace with a compute cluster in your workspace.
with tempfile.TemporaryDirectory() as temp_dir:
    training_data_path = Path(temp_dir) / "iris.csv"
    training_data.to_csv(training_data_path, index=False)

    training_job = command(
        code=str(project_dir),
        command=(
            "python azure_train.py "
            "--training-data ${{inputs.training_data}} "
            "--model-output ${{outputs.model_output}} "
            "--C 1.0 --max-iter 100"
        ),
        inputs={
            "training_data": Input(
                type="uri_file",
                path=str(training_data_path),
            )
        },
        outputs={"model_output": Output(type="uri_folder")},
        environment=sklearn_environment,
        compute=f"azureml:{compute_name}",
        experiment_name="iris-sklearn-classification",
        display_name="train-iris-logistic-regression",
    )
    submitted_job = ml_client.jobs.create_or_update(training_job)

ml_client.jobs.stream(submitted_job.name)

# Register the model from the named job output.
model = Model(
    path=(
        f"azureml://jobs/{submitted_job.name}/outputs/"
        "model_output/paths/"
    ),
    name="iris-sklearn-model",
    type="custom_model",
    description="Iris classifier trained with Scikit-learn.",
)
registered_model = ml_client.models.create_or_update(model)

# Create a uniquely named managed online endpoint and deploy the model.
endpoint_name = f"iris-sklearn-{uuid.uuid4().hex[:8]}"
endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Iris Scikit-learn classification endpoint.",
    auth_mode="key",
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=endpoint_name,
    model=registered_model,
    environment=sklearn_environment,
    code_configuration=CodeConfiguration(
        code=str(project_dir),
        scoring_script="azure_score.py",
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)
try:
    ml_client.online_deployments.begin_create_or_update(deployment).result()

    # Route all endpoint traffic to this deployment.
    endpoint.traffic = {"blue": 100}
    endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
    print(f"Endpoint ready: {endpoint.scoring_uri}")

    # Send a small sample request to verify the deployed model.
    with tempfile.TemporaryDirectory() as request_dir:
        request_path = Path(request_dir) / "sample-request.json"
        request_path.write_text(
            json.dumps({"data": iris.data.iloc[:3].values.tolist()}),
            encoding="utf-8",
        )
        prediction = ml_client.online_endpoints.invoke(
            endpoint_name=endpoint_name,
            request_file=str(request_path),
        )
    print(f"Sample predictions: {prediction}")
finally:
    # Remove the endpoint and its deployment to avoid ongoing charges.
    ml_client.online_endpoints.begin_delete(name=endpoint_name).result()

# 8.4 Converting a TensorFlow model to LiteRT (TensorFlow Lite)

In [ ]:
from pathlib import Path

import numpy as np
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

# Define a simple model for MNIST digit classification.
model = tf.keras.Sequential(
    [
        tf.keras.Input(shape=(28, 28)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(10, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Load and normalize the MNIST images as float32 values in [0, 1].
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

model.fit(x_train, y_train, epochs=5, validation_split=0.2)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc:.4f}")

# Save the native Keras model and convert it to LiteRT/TensorFlow Lite.
model.save("mnist_model.keras")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
Path("mnist_model.tflite").write_bytes(tflite_model)
print("Model successfully converted and saved as mnist_model.tflite.")


def run_litert_inference(model_content, input_data):
    interpreter = Interpreter(model_content=model_content)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_tensor = np.asarray(input_data, dtype=input_details["dtype"])

    interpreter.set_tensor(input_details["index"], input_tensor)
    interpreter.invoke()
    return interpreter.get_tensor(output_details["index"])


test_image = x_test[:1]
tflite_output = run_litert_inference(tflite_model, test_image)
tflite_prediction = int(np.argmax(tflite_output[0]))
actual_label = int(y_test[0])
print(f"LiteRT model prediction: {tflite_prediction}")
print(f"Actual label: {actual_label}")